# Generate LLM Responses → Save for Later (Prolific AI Tasker–ready)

This notebook:
1. Reads generation settings from `config.yaml`  
2. Loads prompts from `prompts.jsonl` (one JSON object per line with a `prompt` field)  
3. Generates responses per prompt for each temperature and completion index  
4. Saves output to a **CSV** file 

In [1]:
import os, json, time, math, yaml, uuid, datetime
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd
from itertools import combinations

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

In [2]:
# Read API tokens for Jupyter Notebook
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Set paths
data_dir = Path('.')
config_path = Path('..') / 'examples' / 'config.yaml'
prompts_path = Path('..') / 'examples' / 'prompts.jsonl'

output_dir = data_dir / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / 'completions.csv'
pairs_csv = output_dir / 'response_pairs.csv'
metadata_path = output_dir / 'run_metadata.json'

In [4]:
# Load config
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg

{'model': 'meta-llama/Llama-3.2-3B',
 'num_completions_per_prompt': 2,
 'temperatures': [0.7, 1.0],
 'max_new_tokens': 512,
 'top_p': 0.9,
 'prolific': {'batch_name': 'HH-RLHF Pairwise Comparison Batch',
  'task_schema': {'task_name': 'Compare two AI-generated responses',
   'task_introduction': '<p>You will see a short prompt (a user question) and two possible AI assistant responses (“Response A” and “Response B”).</p>\n<p>Your job is to choose which response you would prefer to receive from an assistant.</p>\n',
   'task_steps': '<p>Please consider:</p>\n<ul>\n<li>Accuracy and correctness</li>\n<li>Helpfulness and completeness</li>\n<li>Clarity and organization</li>\n<li>Safety and appropriateness</li>\n</ul>\n<p>Pick the response that feels overall better, even if both are imperfect.</p>\n',
   'task_question': 'Pick the response that feels overall better, even if both are imperfect.',
   'tasks_per_group': 10},
  'study_setup': {'internal_name': 'HH-RLHF Pairwise Comparison',
   'd

In [5]:
# Load prompts
with open(prompts_path, 'r') as f:
    prompts = [json.loads(line) for line in f]

prompts

[{'prompt': 'How do I make homemade pasta from scratch?'},
 {'prompt': 'What are the key differences between machine learning and deep learning?'},
 {'prompt': 'Can you explain how photosynthesis works in simple terms?'},
 {'prompt': 'What are some effective strategies for managing stress at work?'}]

In [6]:
# Turn off HF/ipywidgets/tqdm notebook bars to avoid comm errors with Jupyter
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TQDM"] = "1"         
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

try:
    from huggingface_hub.utils.tqdm import disable_progress_bars
    disable_progress_bars()
except Exception:
    pass

In [7]:
# Detect GPU
if torch.cuda.is_available():
    device = "cuda"
    print("✅ Using NVIDIA GPU (CUDA).")
elif torch.backends.mps.is_available():
    device = "mps"
    print("✅ Using Apple Silicon GPU (Metal).")
else:
    device = "cpu"
    print("⚠️ No GPU found — using CPU.")

✅ Using Apple Silicon GPU (Metal).


In [8]:
# Load model
device_map = "auto" if device == "cuda" else None
torch_dtype = torch.float16 if device in ["cuda", "mps"] else torch.float32

tokenizer = AutoTokenizer.from_pretrained(cfg['model'], use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    cfg['model'],
    dtype=torch_dtype,
    device_map=device_map,
    token=os.environ.get("HF_TOKEN")
)

if device_map is None:
    model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
# Fuction to generate responses
def generate_one(prompt: str, temperature: float, max_new_tokens: int, top_p: float) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    gen_out = model.generate(
        **inputs,
        do_sample=True,
        temperature=float(temperature),
        top_p=float(top_p),
        max_new_tokens=int(max_new_tokens),
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(gen_out[0], skip_special_tokens=True)
    return text

In [10]:
cfg

{'model': 'meta-llama/Llama-3.2-3B',
 'num_completions_per_prompt': 2,
 'temperatures': [0.7, 1.0],
 'max_new_tokens': 512,
 'top_p': 0.9,
 'prolific': {'batch_name': 'HH-RLHF Pairwise Comparison Batch',
  'task_schema': {'task_name': 'Compare two AI-generated responses',
   'task_introduction': '<p>You will see a short prompt (a user question) and two possible AI assistant responses (“Response A” and “Response B”).</p>\n<p>Your job is to choose which response you would prefer to receive from an assistant.</p>\n',
   'task_steps': '<p>Please consider:</p>\n<ul>\n<li>Accuracy and correctness</li>\n<li>Helpfulness and completeness</li>\n<li>Clarity and organization</li>\n<li>Safety and appropriateness</li>\n</ul>\n<p>Pick the response that feels overall better, even if both are imperfect.</p>\n',
   'task_question': 'Pick the response that feels overall better, even if both are imperfect.',
   'tasks_per_group': 10},
  'study_setup': {'internal_name': 'HH-RLHF Pairwise Comparison',
   'd

In [11]:
# Generate completions
rows = []
run_id = str(uuid.uuid4())
started_at = datetime.datetime.utcnow().isoformat() + "Z"

for p_idx, prompt in enumerate(prompts):
    prompt_id = f"p{p_idx+1:04d}"
    prompt = prompt['prompt']
    print(f"\n📝 Prompt {p_idx+1}: {prompt}")
    for temp in cfg['temperatures']:
        print(f"  🌡️ Temperature={temp}")
        for c_idx in range(cfg['num_completions_per_prompt']):
            print(f"    ▶️ Generating completion {c_idx+1}")
            t0 = time.time()
            try:
                completion_text = generate_one(
                    prompt=prompt,
                    temperature=temp,
                    max_new_tokens=cfg['max_new_tokens'],
                    top_p=cfg['top_p'],
                )
                # print("="*89)
                # print(f"Completion text: {completion_text}")
                # print("="*89)
            except Exception as e:
                completion_text = f"[GENERATION_ERROR] {type(e).__name__}: {e}"

            duration_s = time.time() - t0
            print(f" done in {duration_s:.2f}s \n")

            rows.append({
                "run_id": run_id,
                "prompt_id": prompt_id,
                "prompt": prompt,
                "model": cfg['model'],
                "temperature": temp,
                "top_p": cfg['top_p'],
                "max_new_tokens": cfg['max_new_tokens'],
                "completion_index": c_idx,
                "response": completion_text,
                "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
                "latency_seconds": round(duration_s, 4),
            })

print("\n✅ Generation complete.")


📝 Prompt 1: How do I make homemade pasta from scratch?
  🌡️ Temperature=0.7
    ▶️ Generating completion 1
 done in 21.68s 

    ▶️ Generating completion 2
 done in 18.99s 

  🌡️ Temperature=1.0
    ▶️ Generating completion 1
 done in 18.99s 

    ▶️ Generating completion 2
 done in 19.05s 


📝 Prompt 2: What are the key differences between machine learning and deep learning?
  🌡️ Temperature=0.7
    ▶️ Generating completion 1
 done in 15.60s 

    ▶️ Generating completion 2
 done in 19.13s 

  🌡️ Temperature=1.0
    ▶️ Generating completion 1
 done in 19.17s 

    ▶️ Generating completion 2
 done in 19.10s 


📝 Prompt 3: Can you explain how photosynthesis works in simple terms?
  🌡️ Temperature=0.7
    ▶️ Generating completion 1
 done in 19.20s 

    ▶️ Generating completion 2
 done in 1.66s 

  🌡️ Temperature=1.0
    ▶️ Generating completion 1
 done in 19.06s 

    ▶️ Generating completion 2
 done in 19.05s 


📝 Prompt 4: What are some effective strategies for managing stress at wor

In [12]:
# Save results into a df
df = pd.DataFrame(rows)

df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Wrote CSV → {csv_path.resolve()} ({len(df)} rows)")

df.head()

Wrote CSV → /Users/vivianamarquez/Documents/github_repos/prolific_ai_taskers/notebooks/outputs/completions.csv (16 rows)


,run_id,prompt_id,prompt,model,temperature,top_p,max_new_tokens,completion_index,response,generated_at,latency_seconds
0,453622a3-5917-4ca9-a2cd-c1ce5cdbf189,p0001,How do I make homemade pasta from scratch?,meta-llama/Llama-3.2-3B,0.7,0.9,512,0,How do I make homemade pasta from scratch? To...,2025-10-24T17:27:28.783552Z,21.6778
1,453622a3-5917-4ca9-a2cd-c1ce5cdbf189,p0001,How do I make homemade pasta from scratch?,meta-llama/Llama-3.2-3B,0.7,0.9,512,1,How do I make homemade pasta from scratch? Mak...,2025-10-24T17:27:47.771090Z,18.9874
2,453622a3-5917-4ca9-a2cd-c1ce5cdbf189,p0001,How do I make homemade pasta from scratch?,meta-llama/Llama-3.2-3B,1.0,0.9,512,0,How do I make homemade pasta from scratch? How...,2025-10-24T17:28:06.765156Z,18.9940
3,453622a3-5917-4ca9-a2cd-c1ce5cdbf189,p0001,How do I make homemade pasta from scratch?,meta-llama/Llama-3.2-3B,1.0,0.9,512,1,How do I make homemade pasta from scratch? How...,2025-10-24T17:28:25.816297Z,19.0511
4,453622a3-5917-4ca9-a2cd-c1ce5cdbf189,p0002,What are the key differences between machine l...,meta-llama/Llama-3.2-3B,0.7,0.9,512,0,What are the key differences between machine l...,2025-10-24T17:28:41.416741Z,15.6003


In [13]:
# Metadata 
meta = {
    "run_id": run_id,
    "started_at": started_at,
    "finished_at": datetime.datetime.utcnow().isoformat() + "Z",
    "config": cfg,
    "num_prompts": len(prompts),
    "total_completions": len(rows),
}

with open(metadata_path, "w") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

meta

{'run_id': '453622a3-5917-4ca9-a2cd-c1ce5cdbf189',
 'started_at': '2025-10-24T17:27:07.105415Z',
 'finished_at': '2025-10-24T17:31:54.396154Z',
 'config': {'model': 'meta-llama/Llama-3.2-3B',
  'num_completions_per_prompt': 2,
  'temperatures': [0.7, 1.0],
  'max_new_tokens': 512,
  'top_p': 0.9,
  'prolific': {'batch_name': 'HH-RLHF Pairwise Comparison Batch',
   'task_schema': {'task_name': 'Compare two AI-generated responses',
    'task_introduction': '<p>You will see a short prompt (a user question) and two possible AI assistant responses (“Response A” and “Response B”).</p>\n<p>Your job is to choose which response you would prefer to receive from an assistant.</p>\n',
    'task_steps': '<p>Please consider:</p>\n<ul>\n<li>Accuracy and correctness</li>\n<li>Helpfulness and completeness</li>\n<li>Clarity and organization</li>\n<li>Safety and appropriateness</li>\n</ul>\n<p>Pick the response that feels overall better, even if both are imperfect.</p>\n',
    'task_question': 'Pick the 

In [14]:
# Keep only the required columns for pairing
df_copy = df[['prompt_id','prompt','response']].copy()

# Cleanup
df_copy["response"] = df_copy.apply(
    lambda x: x["response"].replace(x["prompt"], "", 1).strip()
    if isinstance(x["response"], str) and x["response"].startswith(x["prompt"])
    else x["response"],
    axis=1
)

df_copy.head()

,prompt_id,prompt,response
0,p0001,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs..."
1,p0001,How do I make homemade pasta from scratch?,Making your own pasta dough is easier than you...
2,p0001,How do I make homemade pasta from scratch?,How to make homemade pasta from scratch 1. Mix...
3,p0001,How do I make homemade pasta from scratch?,How to Make Fresh Pasta Dough. Add 2 to 3 tabl...
4,p0002,What are the key differences between machine l...,"What are the advantages of each approach? And,..."


In [20]:
df_copy.head()

,prompt_id,prompt,response
0,p0001,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs..."
1,p0001,How do I make homemade pasta from scratch?,Making your own pasta dough is easier than you...
2,p0001,How do I make homemade pasta from scratch?,How to make homemade pasta from scratch 1. Mix...
3,p0001,How do I make homemade pasta from scratch?,How to Make Fresh Pasta Dough. Add 2 to 3 tabl...
4,p0002,What are the key differences between machine l...,"What are the advantages of each approach? And,..."


In [22]:
# Generate pairs
rows = []
for (pid, prompt), g in df_copy.groupby(['prompt_id', 'prompt'], dropna=False):
    # keep only non-empty string responses
    responses = [r for r in g['response'] if isinstance(r, str) and r.strip()]
    for a, b in combinations(responses, 2):
        rows.append({
            'Prompt': prompt,
            'Response A': a,
            'Response B': b,
        })

pairs_df = pd.DataFrame(rows, columns=['Prompt', 'Response A', 'Response B']).reset_index(drop=True)
pairs_df.head()

,Prompt,Response A,Response B
0,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs...",Making your own pasta dough is easier than you...
1,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs...",How to make homemade pasta from scratch 1. Mix...
2,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs...",How to Make Fresh Pasta Dough. Add 2 to 3 tabl...
3,How do I make homemade pasta from scratch?,Making your own pasta dough is easier than you...,How to make homemade pasta from scratch 1. Mix...
4,How do I make homemade pasta from scratch?,Making your own pasta dough is easier than you...,How to Make Fresh Pasta Dough. Add 2 to 3 tabl...


In [23]:
pairs_df.to_csv(pairs_csv, index=False)
print(f"✅ Wrote CSV → {pairs_csv.resolve()}  ({len(pairs_df)} rows)")

✅ Wrote CSV → /Users/vivianamarquez/Documents/github_repos/prolific_ai_taskers/notebooks/outputs/response_pairs.csv  (24 rows)
